In [ ]:
import numpy as np
import os
import pandas as pd

from tqdm import tqdm

In [ ]:
# Load ed/edstays table
df_edstays = pd.read_csv('data/ed/edstays.csv', dtype={'stay_id': str, 'subject_id': str, 'arrival_transport': str, 'disposition': str, 'hadm_id': str}, index_col='stay_id')
df_edstays['intime'] = pd.to_datetime(df_edstays['intime'])
df_edstays['outtime'] = pd.to_datetime(df_edstays['outtime'])
df_edstays.loc[:, 'los'] = (df_edstays['outtime'] - df_edstays['intime']).dt.total_seconds() / (60*60)

In [ ]:
# Load hosp/patients table
df_patients = pd.read_csv('data/hosp/patients.csv', dtype={'subject_id': str, 'anchor_age': int, 'anchor_year': int}, index_col='subject_id')
dict_patients = df_patients.to_dict(orient='index')

In [ ]:
# Add patient age to ed/edstays table
df_patientdata = pd.merge(df_edstays, df_patients, how='left', left_index=True, right_on='subject_id')
age_col = []
for stay_idx in tqdm(df_edstays.index):
    val_admityear = df_edstays.loc[stay_idx, 'intime'].year

    subject_idx = df_edstays.loc[stay_idx, 'subject_id']
    val_anchoryear = dict_patients[subject_idx]['anchor_year']
    val_anchorage = dict_patients[subject_idx]['anchor_age']

    age_col.append(val_admityear - val_anchoryear + val_anchorage)
df_edstays.loc[:, 'age'] = age_col

In [ ]:
# Load list of hadm_id in icu/icustays
df_icustays = pd.read_csv('data/icu/icustays.csv', dtype={'hadm_id': str})
list_icustays_hadm_id = df_icustays['hadm_id']

In [ ]:
# Relabel ADMITTED disposition to either WARD or ICU
df_edstays['disposition'] = np.where(df_edstays['disposition'] == 'ADMITTED', 'WARD', df_edstays['disposition'])
df_edstays['disposition'] = np.where(df_edstays['hadm_id'].isin(list_icustays_hadm_id), 'ICU', df_edstays['disposition'])

df_edstays

In [ ]:
# Load ed/triage table
df_triage = pd.read_csv('data/ed/triage.csv', index_col='stay_id', dtype={'stay_id': str, 'acuity': str})
df_triage['acuity'] = df_triage['acuity'].fillna('-1').astype(float).astype(int).astype(str)

df_triage

In [ ]:
# Load ed/diagnosis table (only primary diagnosis)
df_diagnosis = pd.read_csv('data/ed/diagnosis.csv', dtype={'stay_id': str, 'seq_num': str, 'icd_code': str, 'icd_version': str}, index_col='stay_id')
df_diagnosis = df_diagnosis[df_diagnosis['seq_num'] == '1'] # primary diagnosis

df_diagnosis

In [ ]:
# Get columns from each dataframes that will be used for the study
df_edstays = df_edstays[['subject_id', 'age', 'arrival_transport', 'disposition', 'los']]
df_triage = df_triage[['acuity']]

In [ ]:
# Merge into one dataframe
df_patientdata = pd.merge(df_edstays, df_triage, how='left', left_index=True, right_index=True)

df_patientdata

In [ ]:
# Get patients in the test set
df_test = pd.read_csv('data/test.csv')
df_test['stay_id'] = df_test['stay_id'].astype(str)

In [ ]:
df_patientdata = df_patientdata.reset_index()
df_patientdata['stay_id'] = df_patientdata['stay_id'].astype(str)

In [ ]:
df_patientdata['in_test'] = df_patientdata['stay_id'].isin(df_test['stay_id'])
df_patientdata = df_patientdata[df_patientdata['in_test']==True]
df_patientdata

In [ ]:
# Filter data with invalid values for the experiments
df_patientdata = df_patientdata[df_patientdata['acuity'].isin(['1', '2', '3', '4', '5'])].copy()
df_patientdata = df_patientdata[df_patientdata['disposition'].isin(['HOME', 'WARD', 'ICU'])].copy()

df_patientdata

In [ ]:
df_patientdata = df_patientdata[['stay_id', 'subject_id', 'acuity', 'disposition', 'los']]

In [ ]:
# Save preprocessed patient records that will be used for the study
outpath = 'data'
if not os.path.exists(outpath):
    os.makedirs(outpath)

df_patientdata.to_csv(f'{outpath}/patient_data.csv', index=False)